# Module 12: Agent Protocols - 03: A2A vs MCP

> **MLCourse - Agentic AI - Agent Patterns**

You already know MCP well -
[`04_crewai/03_flows_and_orchestration/04_mcp_integration`](../../04_crewai/03_flows_and_orchestration/04_mcp_integration/README.md)
covered tools, resources, prompts, transports, and authentication in depth.
Notebooks 01-02 of this module built A2A's core objects and ran a real
exchange. This notebook puts the two side by side, because the question
"MCP or A2A?" comes up the moment you have both available, and the honest
answer is **"probably both, for different edges of the same system."**

No LLM calls, no API key.

### What you will learn

1. The one-sentence distinction, and why it is more than semantics.
2. A structural comparison across the dimensions that actually matter.
3. A worked system that needs both, wired together explicitly.
4. When reaching for A2A is a mistake.

### Setup


In [ ]:
print("Module 12, notebook 03: A2A vs MCP -- no API key needed")


### 1. The one-sentence distinction

> **MCP connects an agent to tools. A2A connects an agent to another agent.**

That sounds almost too simple, but nearly every structural difference between
the two protocols falls out of it. A tool has no autonomy, no memory of past
calls, and no opinion about how to do its job - it is a function with a typed
signature, and MCP's job is to describe that signature so a model can call it
correctly (`04_mcp_integration`, tools/resources/prompts). An **agent** has
autonomy: it can decide *how* to satisfy a request, it can ask a clarifying
question, and it may be running its own multi-step reasoning loop internally
that the caller never sees.

MCP's central object is the **tool call**: name, typed arguments, a single
returned value. A2A's central object is the **Task**: an id, a status that can
sit in `input-required` for as long as needed, and a history of messages. That
difference is why notebook 02's `SkyBookerAgent` could ask "which fare
class?" mid-interaction and MCP's `call_tool()` fundamentally cannot - an MCP
tool call is request/response, full stop.

### 2. Structural comparison

| | MCP | A2A |
|---|---|---|
| Connects | agent ↔ **tool** (or resource, or prompt) | agent ↔ **agent** |
| Callee has autonomy? | no - a tool executes, it does not decide | **yes** - the remote agent decides how to satisfy the request |
| Core object | tool call: name + typed args → one result | **Task**: id + status + message history |
| Interaction shape | request / response | request / response, **or** a multi-turn negotiation (`input-required`) |
| Discovery | `list_tools()` **after** connecting | **Agent Card**, published, checkable **before** connecting |
| Typical trust boundary | your own tool servers, or ones you deliberately integrate (see `mcp_authentication.py`) | **another organisation's** agent by default |
| Covered in | `04_crewai/.../04_mcp_integration` | this module |
| Who decides *how* the work gets done | the calling agent's model, one tool call at a time | the **remote** agent, internally - the caller only sees Task status |

The "who decides how" row is worth sitting with. When your agent calls an MCP
tool, *your* model is the one reasoning about what to call and in what order
- the tool is inert. When your agent delegates a Task to another agent over
A2A, **you hand over that reasoning**. SkyBooker in notebook 02 decided
internally that it needed a fare class before it could book anything; the
`TravelerAgent` never saw that internal decision, only the resulting
`input-required` status. That is a real transfer of control, not just a
different wire format.

### 3. A system that needs both

Consider a trip-planning agent - genuinely realistic, and a good test of the
distinction, because it needs to reach in both directions at once.

```
                         +-------------------------+
                         |   Trip Planning Agent    |
                         +-------------------------+
                          |                        |
                MCP (tools it OWNS)        A2A (agents it does NOT own)
                          |                        |
             +------------+------------+   +-------+--------+
             |                         |   |                |
      [ weather API tool ]    [ currency-convert tool ]  [ SkyBooker agent ]
        (an MCP server              (an MCP server        (a DIFFERENT
         you deployed)               you deployed)          company's agent,
                                                              reached via A2A)
```

The weather lookup and the currency conversion are **tools**: stateless,
no autonomy, and - crucially - you own the server. MCP is the right protocol
because there is nothing to negotiate; you just want a typed function call.

Booking the flight goes to **SkyBooker**, a genuinely separate agent run by a
different company, with its own internal reasoning about seat availability
and fare classes that you neither control nor need to see. A2A is the right
protocol because SkyBooker is not a function - it is a decision-maker you are
delegating to, exactly as notebook 02 modelled.

The trip-planning agent's own orchestration logic (LangGraph, or a CrewAI
`Flow`) is neither of these - it is the same in-process pattern you have used
throughout this course. **MCP and A2A are the two protocols it reaches for at
its edges**, chosen per edge based on the one-sentence rule from section 1:
tool, or agent?

### The decision, made concrete as a function


In [ ]:
def choose_protocol(target_has_autonomy: bool, target_is_external_org: bool,
                    interaction_may_need_negotiation: bool) -> str:
    """A simplified version of the real decision. In practice all three
    signals tend to move together -- an external org's agent is almost
    always autonomous and almost always may need back-and-forth -- but
    it is worth checking each explicitly rather than pattern-matching on
    'it's another company' alone."""
    if target_has_autonomy or interaction_may_need_negotiation:
        return "A2A"
    return "MCP"


cases = [
    ("in-house weather lookup function", False, False, False),
    ("in-house SQL query tool", False, False, False),
    ("in-house re-ranking model wrapped as a tool", False, False, False),
    ("partner airline's booking agent", True, True, True),
    ("partner airline's flight-status LOOKUP endpoint (stateless, typed)", False, True, False),
    ("your own second agent, but on a different team's infra", True, False, True),
]

print(f"{'target':<58} {'protocol':>8}")
print("-" * 68)
for name, autonomy, external, negotiation in cases:
    print(f"{name:<58} {choose_protocol(autonomy, external, negotiation):>8}")


### The row worth explaining: "partner airline's flight-status endpoint"

This is the case people get wrong most often, and the table includes it on
purpose: **"external organisation" alone does not mean A2A.** If the partner
exposes a flight-status lookup that takes a flight number and returns a typed
status with no negotiation, no autonomy, and no multi-turn conversation, it
is structurally a **tool** regardless of who owns the server. MCP (over its
network transports, with the authentication from `mcp_authentication.py`) is
the right fit, not A2A.

The converse row - **"your own second agent, but on a different team's
infra"** - makes the matching point: A2A is not defined by organisational
boundaries. If the callee genuinely decides *how* to do its work and the
interaction may need back-and-forth, that is an A2A shape even when both
agents belong to you.

### 4. When A2A is the wrong call

Reaching for A2A by default is a real mistake, not a hypothetical one, for
reasons that follow directly from what makes it heavier than MCP:

- **It has no typed schema for the request itself.** An MCP tool declares its
  input schema; the model calling it gets validated, structured arguments. An
  A2A Task's first message is closer to a free-text request the remote agent
  has to interpret - notebook 02's `_extract_flight_id()` is doing exactly
  that interpretation by hand. If what you actually want is "call this
  function with these exact typed arguments," A2A adds indirection MCP
  already solves better.
- **It assumes a genuine capability boundary.** Discovery, Agent Cards, and
  capability matching exist to answer "can this unfamiliar agent do what I
  need," a question that is meaningless when you already know exactly what
  the other side supports because you wrote it.
- **Multi-turn negotiation has a cost.** A protocol built around Tasks that
  can sit in `input-required` needs state management, session tracking, and
  timeout/escalation policy on both sides - see
  `14_async_human_approval` for what that machinery looks like when the
  "other party" is a human rather than another agent. If your interaction is
  reliably one request and one response, that machinery is pure overhead.

The short version: **A2A earns its cost when the callee is genuinely
autonomous and the interaction may genuinely need more than one turn.**
Everything else is a tool call, and MCP already does that well.

### Key takeaways

- **MCP connects an agent to tools. A2A connects an agent to another agent.**
  Every structural difference in the comparison table follows from that one
  sentence.
- MCP's unit is a **typed tool call** (request/response). A2A's unit is a
  **Task** with a status machine that supports genuine multi-turn negotiation
  - the concrete difference notebook 02's `input-required` exchange
  demonstrated.
- **Discovery differs in direction and timing**: MCP's `list_tools()` happens
  *after* connecting to a server you already chose to trust; A2A's Agent Card
  is meant to be checked *before* any connection is made at all.
- A real system typically needs **both**, chosen per integration point: tools
  you own or trust go through MCP; genuinely autonomous agents you are
  delegating decisions to go through A2A.
- Reaching for A2A when a typed MCP tool call would do is a real cost, not a
  purely stylistic choice - it adds a schema-free negotiation layer that
  buys you nothing if there is nothing to negotiate.

That completes module 12. Related reading:
[`04_crewai/.../04_mcp_integration`](../../04_crewai/03_flows_and_orchestration/04_mcp_integration/README.md)
for MCP in full depth, and `14_async_human_approval` for a status-machine
negotiation pattern structurally similar to A2A's `input-required`, but with
a human on the other end instead of a second agent.